# Gunter's Space Page — tabla tabular completa (una fila por objeto)

Este notebook junta **todo** lo ya descargado de Gunter's Space Page
(`data/gunter/`) en una sola tabla ancha, **una fila por objeto/lanzamiento
del registro `#satlist`** del sitio.

Objetivos:

1. **Sin clasificar**: a diferencia de `build_gunter_failures.py`, este
   notebook NO clasifica causas (`space_weather`, `power`, …). Solo ordena en
   columnas lo que el sitio ya dice.
2. **Todas las variables disponibles**: campo `#satdescription` (narrativa en
   prosa), bloque `#satdata` (Nation, Operator, Mass, Orbit, …), registro de
   lanzamientos (COSPAR, fecha, sitio, vehículo, remarks) y años mencionados
   en la narrativa.
3. **Rastreable**: cada fila conserva `source_url` y `page_id` → se puede
   volver a la página original.

## Origen y confiabilidad

Toda la data viene de la prosa/HTML de **Gunter's Space Page**
(https://space.skyrocket.de), referencia curada por Gunter Dirk Krebs. No es
telemetría: las fechas son gruesas y las causas son narración de una persona.
Úsala como capa narrativa, no como dato primario (los datos primarios son
OMNI/DONKI/Space-Track).

> Contenido © Gunter Dirk Krebs 1996–2026, Gunter's Space Page
> (https://space.skyrocket.de). Usado bajo los términos de crawl del sitio
> (`robots.txt` permite crawlear; `noai`: no usar para entrenar modelos;
> RAG/summarización solo con atribución y link al original).

In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
from bs4 import BeautifulSoup

def _find_root(start):
    p = Path(start).resolve()
    while not (p / "data" / "gunter" / "meta" / "pages.parquet").exists() and p != p.parent:
        p = p.parent
    return p

ROOT = _find_root(Path.cwd())
DATA = ROOT / "data" / "gunter"

meta = pd.read_parquet(DATA / "meta/pages.parquet")
tables = pd.read_parquet(DATA / "tables.parquet")
failures = pd.read_parquet(DATA / "failures.parquet")

doc_pages = meta[meta["url"].str.contains("/doc_sdat/", regex=False)].copy()
print(f"paginas doc_sdat: {len(doc_pages)} | registros de lanzamiento doc_sdat: "
      f"{(tables["source_url"].isin(doc_pages["url"])).sum()}")

paginas doc_sdat: 6710 | registros de lanzamiento doc_sdat: 32323


In [2]:
ENCODING = "iso-8859-1"
failed_parse = []

def parse_page_html(html: str):
    soup = BeautifulSoup(html, "html.parser")
    desc_el = soup.find(id="satdescription")
    description = ""
    if desc_el is not None:
        paras = [p.get_text(" ", strip=True) for p in desc_el.find_all("p")]
        description = "\n\n".join(paras)
    satdata = {}
    table = soup.find("table", id="satdata")
    if table is not None:
        for tr in table.find_all("tr"):
            th, td = tr.find("th"), tr.find("td")
            if th is not None and td is not None:
                key = th.get_text(" ", strip=True).rstrip(":")
                satdata[key] = td.get_text(" ", strip=True)
    return description, satdata

page_cache = {}
failed_parse = []
for i, row in doc_pages.iterrows():
    f = DATA / "pages" / f"{row['page_id']}.html"
    if not f.exists() or not f.stat().st_size:
        failed_parse.append(row["url"])
        continue
    page_cache[row["url"]] = parse_page_html(f.read_text(errors="replace", encoding=ENCODING))
    if (len(page_cache) in {1000, 3000, 5000, 6500}) or ((i + 1) == len(doc_pages)):
        print(f"parseadas {len(page_cache)}/{len(doc_pages)} paginas")

print("paginas sin parsear:", len(failed_parse))
print("ejemplo metadatos:", page_cache["https://space.skyrocket.de/doc_sdat/galaxy-15.htm"][1])

parseadas 1000/6710 paginas


parseadas 3000/6710 paginas


parseadas 5000/6710 paginas


parseadas 6106/6710 paginas


parseadas 6500/6710 paginas


paginas sin parsear: 0
ejemplo metadatos: {'Nation': 'USA', 'Type / Application': 'Communication', 'Operator': 'PanAmSat', 'Contractors': 'Orbital Sciences Corporation (OSC)', 'Equipment': '20-24 C-band transponders, WAAS payload', 'Configuration': 'Star-2 Bus', 'Propulsion': 'IHI BT-4', 'Power': '2 deployable solar arrays, batteries', 'Lifetime': '15 years', 'Mass': '2033\xa0kg (launch), 885\xa0kg (dry)', 'Orbit': 'GEO'}


In [3]:
pages_records = []
for url, (description, satdata) in page_cache.items():
    row = doc_pages.set_index("url").loc[url]
    pages_records.append({
        "source_url": url,
        "page_id": row["page_id"],
        "title": row["title"],
        "description_text": description,
        "nation": satdata.get("Nation"),
        "type_application": satdata.get("Type / Application"),
        "operator": satdata.get("Operator"),
        "contractors": satdata.get("Contractors"),
        "equipment": satdata.get("Equipment"),
        "configuration": satdata.get("Configuration"),
        "propulsion": satdata.get("Propulsion"),
        "power": satdata.get("Power"),
        "lifetime": satdata.get("Lifetime"),
        "mass": satdata.get("Mass"),
        "orbit": satdata.get("Orbit"),
    })
pages_df = pd.DataFrame(pages_records)
pages_df["mentions_years"] = pages_df["description_text"].apply(
    lambda s: sorted({int(y) for y in re.findall(r"\b(?:19|20)\d{2}\b", str(s))})
)
print(pages_df.shape)

(6710, 16)


In [4]:
def parse_launch_date(s):
    if not isinstance(s, str) or not s.strip() or s.strip() == "-":
        return pd.NaT
    try:
        return pd.to_datetime(s, format="%d.%m.%Y", errors="raise")
    except (ValueError, TypeError):
        return pd.to_datetime(s, errors="coerce", dayfirst=True)

registry = tables[tables["source_url"].isin(page_cache)].copy()
registry = registry.rename(columns={"Satellite": "satellite", "COSPAR": "cospar",
                                    "LS": "launch_site", "Launch Vehicle": "launch_vehicle",
                                    "Remarks": "remarks"})
registry["launch_date"] = registry["Date"].apply(parse_launch_date)
registry["launch_date_raw"] = registry["Date"]

tabla = registry.merge(pages_df, on="source_url", how="left")
tabla = tabla.drop(columns=["Date", ""])
print("filas objeto:", len(tabla))
print(tabla[["satellite", "cospar", "launch_date", "launch_vehicle", "orbit"]].head(3).to_string(index=False))

filas objeto: 32323
                            satellite   cospar launch_date        launch_vehicle                         orbit
Mars Telecommunications Orbiter (MTO)        -  2028-01-01                       Heliocentric, then Mars orbit
                         Slippers2sat 2025-292  2025-12-10 Lijian-1 (Kinetica-1)                              
               AIRSAT 11 (Zhongke 11)        -  2026-01-01                                                    


In [5]:
fail_page = (
    failures
    .groupby("source_url")
    .agg(failure_year=("failure_year", "first"),
         failure_years=("failure_year", lambda s: sorted({int(x) for x in s.dropna()})),
         recovered=("recovered", "any"),
         failure_reason=("reason", lambda s: " | ".join(dict.fromkeys(str(x) for x in s))))
    .reset_index()
)
tabla = tabla.merge(fail_page, on="source_url", how="left")
print("paginas con falla unidas:", int(tabla["failure_reason"].notna().sum()), "| filas totales:", len(tabla))

paginas con falla unidas: 9186 | filas totales: 32323


In [6]:
print("=== Tamaño / cobertura ===")
print("filas objeto:", len(tabla))
print("páginas únicas:", tabla["source_url"].nunique())
print("COSPAR no vacíos:", tabla["cospar"].notna().sum(), "/", len(tabla))
print("con narrativa:", tabla["description_text"].fillna("").ne("").sum())
print("con falla narrada:", tabla["failure_reason"].notna().sum())
print()
print("=== % NaN por columna (top) ===")
for col, pct in tabla.isna().mean().sort_values(ascending=False).head(8).items():
    print(f"  {col:<28} {pct*100:5.1f}%")
print()
print("=== Ejemplo: Galaxy 15 ===")
g = tabla[(tabla["satellite"].str.contains("Galaxy 15", regex=False, na=False) & tabla["source_url"].str.contains("galaxy-15"))].head(1)
print(g[["satellite", "cospar", "launch_date", "operator", "orbit", "failure_year", "recovered"]].T)
print()
print("=== Ejemplo: Telstar 401 ===")
t401 = tabla[tabla["source_url"].str.contains("telstar-4", regex=False)].head(2)
print(t401[["satellite", "cospar", "launch_date", "failure_reason"]].to_string(index=False))

=== Tamaño / cobertura ===
filas objeto: 32323
páginas únicas: 5152
COSPAR no vacíos: 32323 / 32323
con narrativa: 32323
con falla narrada: 9186

=== % NaN por columna (top) ===
  failure_year                  94.5%
  recovered                     71.6%
  failure_reason                71.6%
  failure_years                 71.6%
  launch_date                    9.2%
  type_application               1.1%
  remarks                        0.0%
  launch_site                    0.0%

=== Ejemplo: Galaxy 15 ===
                                   2511
satellite     Galaxy 15 (ex Galaxy 1RR)
cospar                        2005-041A
launch_date         2005-10-13 00:00:00
operator                       PanAmSat
orbit                               GEO
failure_year                     2022.0
recovered                          True

=== Ejemplo: Telstar 401 ===
  satellite    cospar launch_date                                                                                                           

In [7]:
out_parquet = DATA / "gunter_tabular.parquet"
out_csv = DATA / "gunter_tabular.csv"
tabla.to_parquet(out_parquet, index=False)
tabla.to_csv(out_csv, index=False)
print("escrito:", out_parquet, "->", out_parquet.stat().st_size // 1024, "KiB")
print("escrito:", out_csv, "->", out_csv.stat().st_size // 1024, "KiB")

escrito: /home/pxtron/Documents/cme-sentinel/data/gunter/gunter_tabular.parquet -> 8119 KiB
escrito: /home/pxtron/Documents/cme-sentinel/data/gunter/gunter_tabular.csv -> 69669 KiB
